![interpreto_banner](../assets/img/interpreto_banner.png)

# Classification Concept-based Explanation Tutorial

Welcome to this tutorial, our will be to obtain concept-based explanations starting from the beginning.

For any precision, please refer to the [**Interpreto documentation**](https://for-sight-ai.github.io/interpreto/).

There are five key steps for concepts based explanations:

1. [**Split** your model in two parts](#split)
2. [Compute a dataset of **activations**](#activations)
3. [**Fit** a concept model on activations](#fit)
4. [**Interpret** the concept dimensions](#interpret)
5. [Find the globally **important** concepts](#important)

On which we add three bonus steps:

6. [**Class-wise** concepts and LLM label](#class-wise)
7. [**Locally** important concepts](#locally)
8. [**Evaluate** concept-based explanations](#evaluate)

*Author: Antonin Poché*

In [1]:
import torch

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")

## 1. **Split** your model in two parts <a class="anchor" id="split"></a>

We choose a `DistilBERT` fine-tuned on the `AG-News` dataset and split it just before the classification head.

To split the model, we use the [`interpreto.ModelWithSplitPoints`](https://for-sight-ai.github.io/interpreto/api/concepts/model_with_split_points/) which wraps around the `transformers` model and allows the computation of activations at the specified `split_points`.

In [2]:
from transformers import AutoModelForSequenceClassification

from interpreto import ModelWithSplitPoints

model_with_split_points = ModelWithSplitPoints(
    model_or_repo_id="textattack/distilbert-base-uncased-ag-news",
    automodel=AutoModelForSequenceClassification,
    split_points=[5],  # split at the sixth layer
    device_map="cuda",
    batch_size=1024,
)

## 2. Compute a datasets of **activations** <a class="anchor" id="activations"></a>

We load the first 10000 documents of the `AG-News` train set.

Then we extract the activations of the [CLS] token of each document.

> **Common practice**
>
> In the literature, to train concepts for classification it is common to use the [CLS] just before the classification head.
>
> In fact, at this layer, it makes no sense to use other elements.

> **Warning**
>
> In this notebook, many things are specific to the use of the [CLS] token.

[`interpreto.ModelWithSplitPoints.get_activations()`](https://for-sight-ai.github.io/interpreto/api/concepts/model_with_split_points/#interpreto.ModelWithSplitPoints.get_activations)

In [ ]:
from datasets import load_dataset

# load the AG-News dataset
dataset = load_dataset("fancyzhx/ag_news")
inputs = dataset["train"]["text"][:1000]  # here we use only 1000 examples to go faster, but the more, the better
classes_names = dataset["train"].features["label"].names

# Compute the [CLS] token activations
granularity = ModelWithSplitPoints.activation_granularities.CLS_TOKEN
activations = model_with_split_points.get_activations(
    inputs=inputs,
    activation_granularity=granularity,
    tqdm_bar=True,
    include_predicted_classes=True,
)

Computing activations: 100%|██████████| 1/1 [00:02<00:00,  2.61s/batch]


## 3. **Fit** a concept model on activations <a class="anchor" id="fit"></a>

With activations, we can train a concept model to find patterns (concepts).

The `concept_model` is an attribute of our concept explainer, similarly to the `model_with_split_points`. With these these two elements, we can go from inputs to concepts and from concepts to outputs.

In this tutorial, we use [`interpreto.concepts.ICAConcepts`](https://for-sight-ai.github.io/interpreto/api/concepts/methods/optim/#interpreto.concepts.ICAConcepts) built upon the ICA (Independent Component Analysis) dimension reduction algorithm.

There are at least 15 others concept model available in interpreto. do not hesitate to explore them.

> **Tip**
>
> `ICAConcepts` is a good first candidate for classification. It has no requirements and show good performances on most datasets.

In [4]:
from interpreto.concepts import ICAConcepts

# instantiate the concept explainer
concept_explainer = ICAConcepts(model_with_split_points, nb_concepts=50, device="cuda")

# fit the concept explainer on activations
concept_explainer.fit(activations)

## 4. **Interpret** the concept dimensions <a class="anchor" id="interpret"></a>

We have our concepts and the link between concepts and classes. But now, we need to make sense of these concepts.

In this case, we will use the [`interpreto.concepts.interpretations.TopKInputs`](https://for-sight-ai.github.io/interpreto/api/concepts/concepts_interpretations/#interpreto.concepts.interpretations.TopKInputs) to find the 8 words which activates the most our concepts.

> **Warning**
>
> If the `granularity` specified to the interpretation method is not the same as the one used for activations, the results will be wrong.

In [5]:
from interpreto.concepts.interpretations import TopKInputs

# instantiate the interpretation method with the concept explainer
topk_inputs_method = TopKInputs(
    concept_explainer=concept_explainer,
    k=5,
    activation_granularity=granularity,
    use_unique_words=True,  # with the [CLS] token granularity, we are forced to use unique words
    unique_words_kwargs={
        "count_min_threshold": round(len(inputs) * 0.002),  # appear in at least 0.2% of the samples | increase if random words appear and decrease if some words appear too often
        "lemmatize": True,
        "words_to_ignore": [],  # include noise words and punctuation
    },
)

In [6]:
# call the interpretation methods on the inputs
# we cannot give the previously computed activations because `use_unique_words=True` creates samples with a single word inside
topk_words = topk_inputs_method.interpret(
    inputs=inputs,
    concepts_indices="all",
)

## 5. Find the globally **important** concepts <a class="anchor" id="important"></a>

We have concept directions, it means that our model has access to them, but not that it uses them.

It is the same when you train a model on tabular data, not all features are used.

In this step, we use the [`ConceptAutoEncoderExplainer.concept_output_gradients`](https://for-sight-ai.github.io/interpreto/api/concepts/methods/base/#interpreto.concepts.ConceptAutoEncoderExplainer.concept_output_gradient) to evaluate the importance of each concept with respect to the predicted classes.

> **Note**
>
> All unsupervised concept-based explainers in Interpreto inherit from [`ConceptAutoEncoderExplainer`](https://for-sight-ai.github.io/interpreto/api/concepts/methods/base/#interpreto.concepts.ConceptAutoEncoderExplainer).

> **Note 2**
>
> This step can be done prior to the interpretation, as the interpretation step can be compute heavy. Then specify using the `concept_indices` parameter.
> Only interpreting the important concepts can be wise. (Here we only have 50 concepts, so it does not matter.)

In [7]:
import torch

# estimate the importance of concepts for each class using the gradient
gradients = concept_explainer.concept_output_gradient(
    inputs=inputs,
    targets=None,  # None means all classes
    activation_granularity=granularity,
    concepts_x_gradients=True,  # the concept to output gradients are multiplied by the concepts values, this is common practice in the literature
    batch_size=64,
)

# stack gradients on samples and average them over samples
mean_gradients = torch.stack(gradients).abs().squeeze().mean(0)  # (num_classes, num_concepts)

# for each class, sort the importance scores
order = torch.argsort(mean_gradients, descending=True)

for target in range(order.shape[0]):
    print(f"\nClass: {classes_names[target]}:")
    for i in range(5):
        concept_id = order[target, i].item()
        importance = mean_gradients[target, concept_id].item()
        words = list(topk_words.get(concept_id, None).keys())
        print(f"\tconcept id: {concept_id},\timportance: {round(importance, 3)},\ttopk words: {words}")


Class: World:
	concept id: 27,	importance: 0.091,	topk words: ['serbia-montenegro', 'nato', 'liechtenstein', 'nikkei', 'ossetia']
	concept id: 45,	importance: 0.078,	topk words: ['separatist', 'militia', 'militiaman', 'usatoday.com', 'inquirer']
	concept id: 11,	importance: 0.07,	topk words: ['betting', 'saudi', 'gambler', 'shark', 'kidnapper']
	concept id: 31,	importance: 0.064,	topk words: ['betting', 'fraud', 'holy', 'kmart', 'p.m.']
	concept id: 48,	importance: 0.041,	topk words: ['afp', 'armed', 'hue', 'anarchist', 'naval']

Class: Sports:
	concept id: 7,	importance: 0.111,	topk words: ['phelps', '200-meter', 'batter', 'inning', 'homered']
	concept id: 49,	importance: 0.092,	topk words: ['heat', '100-meter', 'fastest', '200-meter', '200m']
	concept id: 30,	importance: 0.087,	topk words: ['200-meter', '100-meter', 'breaststroke', '400-meter', 'heat']
	concept id: 33,	importance: 0.069,	topk words: ['phillies', 'mets', 'baltimore', 'sox', 'nl']
	concept id: 24,	importance: 0.062,	t

> **The concepts are not interpretable, what do I do?**
>
> - Try to improve the concept-space:
>   - Increases the number of samples. You can artificially do so by splitting then by sentences (not included)
>   - Try different concept-models and parameters
>   - Try to compute concepts class-wise see [next section](#class-wise)
>
> - Improve the interpretation of concepts:
>   - Play with the parameters
>   - Try [`LLMLabels`](https://for-sight-ai.github.io/interpreto/api/concepts/concepts_interpretations/#interpreto.concepts.interpretations.LLMLabels) see [next section](#class-wise)
>
> - Try to evaluate the concepts, to automatically find the best methods. Check this other tutorial: [TODO](TODO)
>
> - Never forget the **faithfulness-plausibility trade-off** of explanations

## 6. Better concepts with class-wise concepts and LLM labels <a class="anchor" id="class-wise"></a>

This section aims at improving the concepts learned by the model. We try three different approaches:

- Training class-wise concepts
- Using another concept model: [`SemiNMFConcepts`](https://for-sight-ai.github.io/interpreto/api/concepts/methods/optim/#interpreto.concepts.SemiNMFConcepts)
- Using LLM labels to interpret the concepts

When a single concept-space is defined for all classes, concepts tend to correspond to the classes themselves. In particular, when the concept-space is built upon on the latent space just before the classification head.

In this section, we will learn a concept space for each class separately. Thus, the class-wise concept explainers will only see examples from a single class (based on the predictions).

In [ ]:
import os

from interpreto.concepts import LLMLabels, SemiNMFConcepts
from interpreto.model_wrapping.llm_interface import OpenAILLM

# Load API key from environment variable
api_key = os.getenv("OPENAI_API_KEY")

# set the LLM interface used to generate labels based on the constructed prompts
llm_interface = OpenAILLM(api_key=api_key, model="gpt-4.1-nano")

concept_explainers = {}
concept_interpretations = {}
concept_importances = {}

# iterate over classes
for target, class_name in enumerate(classes_names):
    # ----------------------------------------------------------------------------------------------
    # 2. construct the dataset of activations (extract the ones related to the class)
    indices = (activations["predictions"] == target).nonzero(as_tuple=True)[0]
    class_wise_inputs = [inputs[i] for i in indices]
    class_wise_activations = {k: v[indices] for k, v in activations.items()}

    # ----------------------------------------------------------------------------------------------
    # 3. train concept model
    concept_explainers[target] = SemiNMFConcepts(model_with_split_points, nb_concepts=20, device="cuda")
    concept_explainers[target].fit(class_wise_activations)

    # ----------------------------------------------------------------------------------------------
    # 5. compute concepts importance (before interpretations to limit the number of concepts interpreted)
    gradients = concept_explainers[target].concept_output_gradient(
        inputs=class_wise_inputs,
        targets=[target],
        activation_granularity=granularity,
        concepts_x_gradients=True,
        batch_size=64,
    )

    # stack gradients on samples and average them over samples
    concept_importances[target] = torch.stack(gradients, axis=0).squeeze().abs().mean(dim=0)  # (num_concepts,)

    # for each class, sort the importance scores
    important_concept_indices = torch.argsort(concept_importances[target], descending=True)[:5].tolist()

    # ----------------------------------------------------------------------------------------------
    # 4. interpret the important concepts concepts
    llm_labels_method = LLMLabels(
        concept_explainer=concept_explainers[target],
        activation_granularity=granularity,
        llm_interface=llm_interface,
        k_examples=10,
        use_unique_words=True,
        unique_words_kwargs={
            "count_min_threshold": round(len(class_wise_inputs) * 0.005),  # appear in at least 0.5% of the samples
        },
    )

    concept_interpretations[target] = llm_labels_method.interpret(
        inputs=class_wise_inputs,
        concepts_indices=important_concept_indices,  # only the top 5 concepts for this class
    )

    print(f"\nClass: {class_name}")
    for concept_id in important_concept_indices:
        label = concept_interpretations[target].get(concept_id.item(), None)
        importance = concept_importances[target][concept_id].item()
        if label is not None:
            print(f"\timportance: {round(importance, 3)},\t{label}")


Class: World
	importance: 0.135,	Geopolitical and political entity references.
	importance: 0.103,	Use of abbreviations, proper nouns, and luxury or social-related terms.
	importance: 0.101,	Groupings of collective identities and associated social issues.
	importance: 0.079,	Capitalized or hyphenated compound keywords related to sports, politics, or events.
	importance: 0.078,	Religious and spiritual terminology dominance

Class: Sports
	importance: 0.134,	Concise sporting or scoring terminology.
	importance: 0.114,	Numeric indicators combining words and hyphens often denote ordinal positions or specific references.
	importance: 0.101,	Golf and baseball terminology with hyphenated and compound forms.
	importance: 0.099,	Sports team and game sequence keywords
	importance: 0.093,	Consistent use of lowercase, punctuation, and hyphenation patterns

Class: Business
	importance: 0.112,	Concatenated corporate and industry terminology patterns
	importance: 0.106,	Commercial language focusing 

## 6. **Locally** important concepts <a class="anchor" id="locally"></a>

In [10]:
test_examples = dataset["test"]["text"][:50]
test_labels = dataset["test"]["label"][:50]
test_preds = model_with_split_points.get_activations(
    inputs=test_examples,
    activation_granularity=granularity,
    tqdm_bar=True,
    include_predicted_classes=True,
)["predictions"]

for class_id, class_name in enumerate(classes_names):

    # extract example
    example_id = test_labels.index(class_id)
    example = test_examples[example_id]
    pred = test_preds[example_id].item()
    print(f"Local importance for example of class {class_name} (pred: {classes_names[pred]}):")
    print(f"Example: {example}")

    # compute local concepts importance for the class
    local_importance = concept_explainer.concept_output_gradient(
        inputs=[example],
        activation_granularity=granularity,
        concepts_x_gradients=True,
        tqdm_bar=False,
    )[0][class_id, 0]

    # normalize local importance and sort it
    normalized_importance = local_importance.abs() / local_importance.abs().sum()
    ordered_indices = torch.argsort(normalized_importance, descending=True)

    # print top 5 concepts
    for concept_id in ordered_indices[:5]:
        importance = normalized_importance[concept_id]
        words_importance = topk_words[concept_id.item()]
        if words_importance is not None:
            print(f"\t{concept_id}: {round(importance.item(), 3)} - {list(words_importance.keys())}")
        else:
            print(f"\t{concept_id}: {round(importance.item(), 3)} - None")

    print("\n")


Computing activations: 100%|██████████| 1/1 [00:00<00:00, 11.90batch/s]

Local importance for example of class World (pred: World):
Example: Sister of man who died in Vancouver police custody slams chief (Canadian Press) Canadian Press - VANCOUVER (CP) - The sister of a man who died after a violent confrontation with police has demanded the city's chief constable resign for defending the officer involved.


	11: 0.347 - ['betting', 'saudi', 'gambler', 'shark', 'kidnapper']
	27: 0.126 - ['serbia-montenegro', 'nato', 'liechtenstein', 'nikkei', 'ossetia']
	31: 0.062 - ['betting', 'fraud', 'holy', 'kmart', 'p.m.']
	6: 0.058 - ['typhoon', 'hurricane', 'earthquake', 'landslide', 'hit']
	48: 0.047 - ['afp', 'armed', 'hue', 'anarchist', 'naval']


Local importance for example of class Sports (pred: Sports):
Example: Giddy Phelps Touches Gold for First Time Michael Phelps won the gold medal in the 400 individual medley and set a world record in a time of 4 minutes 8.26 seconds.
	30: 0.522 - ['200-meter', '100-meter', 'breaststroke', '400-meter', 'heat']
	7: 0.124 - ['phelps', '200-meter', 'batter', 'inning', 'homered']
	35: 0.076 - ['realignment', 'dallas', 'hunger-striking', 'playboy', 'goodale']
	24: 0.072 - ['stryker', 'nfl', 'armadillo', 'homer', 'autodesk']
	16: 0.034 - ['stamps.com', 'missile-defense', 'philadelphia', 'trade', 'israeli']


Local importance for example of class Business (pred

## 7. **Evaluate** concept-based explanations <a class="anchor" id="evaluate"></a>

### 7.1 Evaluate the concept-space from the [third part](#fit)

### 7.2 Evaluate the concepts-interpretations from the [fifth step](#important)

### 7.3 Evaluate the whole concept-based explanations with `ConSim`

In [11]:
# Define the User-LLM (the meta-predictor and llm as a judge)
user_llm = OpenAILLM(api_key="YOUR_OPENAI_API_KEY", model="gpt-4.1-nano")

# Initialize the ConSim  with the model with split points and the user-llm
# Therefore, a given ConSim metric can be used on different explainers for cleaner comparison
con_sim = ConSim(model_with_split_points, user_llm, classes=classes)

# Select examples for evaluation
samples, labels, predictions = con_sim.select_examples(
    dataset["train"]["text"],
    dataset["train"]["label"],
)

# Compute a baseline and ConSim score to give sense to the explainer ConSim score
baseline = con_sim.evaluate(samples, labels, predictions, prompt_type=PromptTypes.L2_baseline_with_lp)

# Compute the ConSim score for an explainer # TODO: allow to give a list
con_sim_score = con_sim.evaluate(
    samples, labels, predictions, concept_explainer, prompt_type=PromptTypes.E3_global_and_local_concepts_with_lp
)

NameError: name 'ConSim' is not defined

In [ ]:
activations.keys()